# Philadelphia Restaurant Data Analysis (Merged & Refined)

This notebook explores the Philadelphia restaurant dataset (20+ reviews, 2019+ reviews only).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

## 1. Load Data

Loading the latest filtered datasets: businesses (20+ reviews, open) and reviews (2019+).

In [ ]:
biz_path = Path("../data/processed/philly_restaurants.csv")
reviews_path = Path("../data/processed/philly_reviews.csv")

df_biz = pd.read_csv(biz_path)
df_reviews = pd.read_csv(reviews_path)
df_reviews['date'] = pd.to_datetime(df_reviews['date'])
df_reviews['year'] = df_reviews['date'].dt.year

print(f"Businesses loaded: {len(df_biz)}")
print(f"Reviews loaded: {len(df_reviews)}")

## 2. Statistical Overview

Summary of star ratings and review counts.

In [ ]:
print("=== Business Metadata Statistics ===")
display(df_biz[['stars', 'review_count']].describe())

mean_stars = df_biz['stars'].mean()
print(f"\nMean Stars: {mean_stars:.2f}")

## 3. Rating Distributions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Overall Restaurant Ratings
sns.countplot(data=df_biz, x='stars', color='skyblue', ax=ax[0])
ax[0].set_title('Distribution of Restaurant Star Ratings')
ax[0].set_xlabel('Stars')
ax[0].set_ylabel('Count')

# Individual Review Ratings
sns.countplot(data=df_reviews, x='stars', palette="rocket", ax=ax[1])
ax[1].set_title('Distribution of Individual Review Ratings')
ax[1].set_xlabel('Stars')
ax[1].set_ylabel('Count')

plt.show()

## 4. Dataset Patterns: Neighborhoods & Cuisines

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 7))

# Top Neighborhoods
if 'neighborhood' in df_biz.columns and df_biz['neighborhood'].notna().any():
    top_neighborhoods = df_biz['neighborhood'].value_counts().head(10)
    sns.barplot(x=top_neighborhoods.values, y=top_neighborhoods.index, palette="viridis", ax=ax[0])
    ax[0].set_title('Top 10 Neighborhoods by Restaurant Count')
    ax[0].set_xlabel('Number of Restaurants')

# Top specific cuisine types
all_categories = []
for cats in df_biz['categories'].dropna():
    all_categories.extend([c.strip() for c in cats.split(',')])
cat_counts = Counter(all_categories)
for generic in ['Restaurants', 'Food']: 
    if generic in cat_counts: del cat_counts[generic]
top_cats = pd.Series(dict(cat_counts)).sort_values(ascending=False).head(15)
sns.barplot(x=top_cats.values, y=top_cats.index, palette="magma", ax=ax[1])
ax[1].set_title('Top 15 Specific Cuisine Categories')
ax[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 5. Trends over Time

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

# Yearly Breakdown (historical perspective)
review_counts_by_year = df_reviews.groupby('year').size()
review_counts_by_year.plot(kind='bar', color='coral', ax=ax[0])
ax[0].set_title('Reviews per Year (2019-Present)')
ax[0].set_xlabel('Year')
ax[0].set_ylabel('Number of Reviews')

# Monthly Trend (finer granularity)
df_reviews['month_year'] = df_reviews['date'].dt.to_period('M')
monthly_reviews = df_reviews.groupby('month_year').size()
monthly_reviews.plot(marker='o', color='teal', ax=ax[1])
ax[1].set_title('Monthly Review Volume (2019 - Present)')
ax[1].set_ylabel('Number of Reviews')

plt.show()

## 6. Price & Sentiment Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Stars by Price Range
if 'price_range' in df_biz.columns:
    df_biz['price_range'] = pd.to_numeric(df_biz['price_range'], errors='coerce')
    sns.boxplot(data=df_biz.dropna(subset=['price_range']), x='price_range', y='stars', palette="muted", ax=ax[0])
    ax[0].set_title('Restaurant Stars by Price Range')
    ax[0].set_xlabel('Price Range (1=$ to 4=$$$$)')

# Useful Votes by Rating
avg_useful = df_reviews.groupby('stars')['useful'].mean()
sns.barplot(x=avg_useful.index, y=avg_useful.values, palette="coolwarm", ax=ax[1])
ax[1].set_title('Average "Useful" Votes by Star Rating')
ax[1].set_ylabel('Mean Useful Votes')

plt.show()